# DegradationSurrogate1DCNN — Battery Capacity Prediction

**Task:** Given one full charge-cycle [V, I, T] time-series → predict next-cycle discharge capacity

## What was wrong in CNN.ipynb

| Issue | Fix applied here |
|---|---|
| Single channel `(1, 180)` — V/I/T concatenated as one flat signal | **3 channels `(3, 60)`** — V, I, T as separate CNN channels |
| No BatchNorm — unstable gradient flow | **BatchNorm1d** after every Conv block |
| No validation split — no early-stopping signal | 80/20 fit/val split + **early stopping** (patience=15) |
| No normalisation — voltage ~3.5 V vs temp ~30 °C | **Per-channel z-score** normalisation (fit on train only) |
| Only training loss reported | **MAE, RMSE, R²** on validation + primary test + secondary test |
| No LR scheduling | **ReduceLROnPlateau** on val loss |
| Named `HealthAgent1DCNN` — wrong role | Renamed **`DegradationSurrogate1DCNN`** |


## 1 · Imports

In [56]:
import numpy as np
import pickle, os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split
from sklearn.metrics import r2_score
import warnings; warnings.filterwarnings('ignore')

BASE_DIR  = r'/Users/mahizhan/Documents/Sem7/Github/data-driven-prediction-of-battery-cycle-life-before-capacity-degradation'
MODEL_DIR = os.path.join(BASE_DIR, 'Model')
os.makedirs(MODEL_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
np.random.seed(42)
print(f'Device: {device}  |  PyTorch {torch.__version__}')
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
import random
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False



Device: cpu  |  PyTorch 2.14.0


## 2 · Load & Merge All Batches (Severson 2019)

In [57]:
print('Loading batch1...')
batch1 = pickle.load(open(os.path.join(BASE_DIR, 'batch1.pkl'), 'rb'))
for k in ['b1c8', 'b1c10', 'b1c12', 'b1c13', 'b1c22']: del batch1[k]

print('Loading batch2...')
batch2 = pickle.load(open(os.path.join(BASE_DIR, 'batch2.pkl'), 'rb'))
b2k = ['b2c7','b2c8','b2c9','b2c15','b2c16']
b1k = ['b1c0','b1c1','b1c2','b1c3','b1c4']
add_len = [662, 981, 1060, 208, 482]
for i, bk in enumerate(b1k):
    batch1[bk]['cycle_life'] += add_len[i]
    for j in batch1[bk]['summary']:
        if j == 'cycle':
            batch1[bk]['summary'][j] = np.hstack((
                batch1[bk]['summary'][j],
                batch2[b2k[i]]['summary'][j] + len(batch1[bk]['summary'][j])))
        else:
            batch1[bk]['summary'][j] = np.hstack((
                batch1[bk]['summary'][j], batch2[b2k[i]]['summary'][j]))
    lc = len(batch1[bk]['cycles'])
    for j, jk in enumerate(batch2[b2k[i]]['cycles']):
        batch1[bk]['cycles'][str(lc + j)] = batch2[b2k[i]]['cycles'][jk]
for k in b2k: del batch2[k]

print('Loading batch3...')
batch3 = pickle.load(open(os.path.join(BASE_DIR, 'batch3.pkl'), 'rb'))
for k in ['b3c37','b3c2','b3c23','b3c32','b3c42','b3c43']: del batch3[k]

numBat1, numBat2, numBat3 = len(batch1), len(batch2), len(batch3)
numBat   = numBat1 + numBat2 + numBat3
bat_dict = {**batch1, **batch2, **batch3}
all_keys = list(bat_dict.keys())

# Official Severson 2019 split
test_ind           = np.hstack((np.arange(0, numBat1 + numBat2, 2), 83))
train_ind          = np.arange(1, numBat1 + numBat2 - 1, 2)
secondary_test_ind = np.arange(numBat - numBat3, numBat)

train_keys    = [all_keys[i] for i in train_ind          if i < len(all_keys)]
pri_test_keys = [all_keys[i] for i in test_ind           if i < len(all_keys)]
sec_test_keys = [all_keys[i] for i in secondary_test_ind if i < len(all_keys)]
print(f'Total:{numBat}  Train:{len(train_keys)}  Pri-test:{len(pri_test_keys)}  Sec-test:{len(sec_test_keys)}')


Loading batch1...
Loading batch2...
Loading batch3...
Total:124  Train:41  Pri-test:43  Sec-test:40


## 3 · Feature Extraction — 3-Channel, 60-Point Interpolation

Each cycle → `(3, 60)` tensor: **V**, **I**, **T** as separate channels (60 pts each).
Target: discharge capacity of the **next** cycle.

**Why 3 separate channels?**
> Conv1d kernels can now learn *channel-specific* patterns: a voltage plateau kernel,
> a temperature-spike kernel — impossible when V/I/T are flattened into one mixed signal.


In [58]:
NUM_PTS = 60  # timesteps per channel

def extract_3ch(cell_dict, key_list, min_cycle=5):
    """
    Returns:
        X    : np.array (N, 3, 60)  -- [V_ch, I_ch, T_ch]
        y    : np.array (N,)         -- next-cycle discharge capacity (Ah)
        meta : list of (cell_id, cycle_num)
    """
    X_list, y_list, meta = [], [], []
    for key in key_list:
        cell   = cell_dict[key]
        cycles = cell['cycles']
        QD     = cell['summary']['QD']
        n_cyc  = len(QD)
        for ci in range(min_cycle, n_cyc - 1):  # need ci+1 for target
            c_str = str(ci)
            if c_str not in cycles: continue
            c = cycles[c_str]
            try:
                V_raw = np.array(c['V'], dtype=np.float32)
                I_raw = np.array(c['I'], dtype=np.float32)
                T_raw = np.array(c['T'], dtype=np.float32)
            except Exception:
                continue
            if len(V_raw) < 4: continue
            g_raw = np.linspace(0, 1, len(V_raw))
            g_fix = np.linspace(0, 1, NUM_PTS)
            V_f = np.interp(g_fix, g_raw, V_raw)
            I_f = np.interp(g_fix, g_raw, I_raw)
            T_f = np.interp(g_fix, g_raw, T_raw)
            X_list.append(np.stack([V_f, I_f, T_f], axis=0))  # (3, 60)
            y_list.append(float(QD[ci + 1]))                   # next-cycle capacity
            meta.append((key, ci))
    return (np.array(X_list, dtype=np.float32),
            np.array(y_list,  dtype=np.float32), meta)

print('Extracting features...')
X_train, y_train, _ = extract_3ch(bat_dict, train_keys)
X_pri,   y_pri,   _ = extract_3ch(bat_dict, pri_test_keys)
X_sec,   y_sec,   _ = extract_3ch(bat_dict, sec_test_keys)
print(f'Train:    {X_train.shape}  y:{y_train.shape}')
print(f'Pri-test: {X_pri.shape}')
print(f'Sec-test: {X_sec.shape}')


Extracting features...
Train:    (27934, 3, 60)  y:(27934,)
Pri-test: (30823, 3, 60)
Sec-test: (41000, 3, 60)


## 4 · Per-Channel Z-Score Normalisation

Voltage ≈ 3.5 V · Current ≈ 3–4 A · Temperature ≈ 25–40 °C — very different scales.
Without normalisation the CNN is dominated by whichever channel has the largest variance.

**Statistics computed on training set only** (no test-set leakage).


In [59]:
# Per-channel mean/std — shape (1, 3, 1) for broadcasting
ch_mean = X_train.mean(axis=(0, 2), keepdims=True)   # (1, 3, 1)
ch_std  = X_train.std( axis=(0, 2), keepdims=True) + 1e-8

def norm_X(X): return (X - ch_mean) / ch_std

X_train_n = norm_X(X_train)
X_pri_n   = norm_X(X_pri)
X_sec_n   = norm_X(X_sec)

# Normalise targets too (loss scale-independent)
y_mean = float(y_train.mean())
y_std  = float(y_train.std()) + 1e-8
def norm_y(y):   return (y - y_mean) / y_std
def denorm_y(y): return y * y_std + y_mean

y_train_n = norm_y(y_train)
y_pri_n   = norm_y(y_pri)
y_sec_n   = norm_y(y_sec)

for ch, name in enumerate(['V', 'I', 'T']):
    print(f'  Channel {name}: mean={ch_mean[0,ch,0]:.4f}  std={ch_std[0,ch,0]:.4f}')
print(f'  Target y: mean={y_mean:.4f} Ah  std={y_std:.4f} Ah')


  Channel V: mean=3.0992  std=0.5493
  Channel I: mean=0.1580  std=3.2148
  Channel T: mean=33.1872  std=2.5236
  Target y: mean=1.0331 Ah  std=0.0569 Ah


## 5 · DataLoaders

80 % of training data for fitting, 20 % held out as **validation** (drives LR scheduler + early stopping).


In [60]:
BATCH_SIZE = 256

full_ds = TensorDataset(
    torch.FloatTensor(X_train_n),
    torch.FloatTensor(y_train_n)
)
n_val = int(len(full_ds) * 0.2)
n_fit = len(full_ds) - n_val
fit_ds, val_ds = random_split(
    full_ds, [n_fit, n_val],
    generator=torch.Generator().manual_seed(42)
)


fit_loader = DataLoader(fit_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader = DataLoader(val_ds,  batch_size=512,        shuffle=False, num_workers=0)
pri_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_pri_n), torch.FloatTensor(y_pri_n)),
    batch_size=512, shuffle=False)
sec_loader = DataLoader(
    TensorDataset(torch.FloatTensor(X_sec_n), torch.FloatTensor(y_sec_n)),
    batch_size=512, shuffle=False)

print(f'Fit samples: {n_fit:,}   Val samples: {n_val:,}')
print(f'Fit batches: {len(fit_loader)}   Val batches: {len(val_loader)}')


Fit samples: 22,348   Val samples: 5,586
Fit batches: 88   Val batches: 11


## 6 · DegradationSurrogate1DCNN Architecture

```
Input  (B, 3, 60)   ← 3 channels × 60 timesteps

Block 1  Conv1d(3→32, k=5, pad=2)  BatchNorm  ReLU  MaxPool(2)         → (B, 32, 30)
Block 2  Conv1d(32→64, k=3, pad=1) BatchNorm  ReLU  MaxPool(2)         → (B, 64, 15)
Block 3  Conv1d(64→128,k=3, pad=1) BatchNorm  ReLU  AdaptiveAvgPool(4) → (B, 128, 4)

Flatten  → 512
Linear(512→256) ReLU Dropout(0.3)
Linear(256→128) ReLU Dropout(0.2)
Linear(128→1)
```

`AdaptiveAvgPool1d(4)` in the last block makes the model robust to slight input-length variations.


In [61]:
class DegradationSurrogate1DCNN(nn.Module):
    """
    3-channel 1D-CNN Degradation Surrogate.
    Input:  (batch, 3, 60)  -- [V, I, T] as separate channels
    Output: (batch, 1)      -- predicted next-cycle discharge capacity (normalised)

    Differences from old HealthAgent1DCNN in CNN.ipynb:
      - 3-channel input instead of flat single-channel concatenation
      - BatchNorm1d after every Conv (stable gradients)
      - AdaptiveAvgPool in final block (length-robust)
      - Kaiming / Xavier weight init
      - Correct semantics: this is a degradation surrogate, not a health agent
    """
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            # Block 1 — short-range waveform features per channel
            nn.Conv1d(3, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),       # (B,32,30)
            # Block 2 — medium-range cross-channel patterns
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),       # (B,64,15)
            # Block 3 — global context, adaptive pooling for robustness
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.AdaptiveAvgPool1d(4),                               # (B,128,4)
        )
        self.head = nn.Sequential(
            nn.Flatten(),           # 128*4 = 512
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 1),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.head(self.conv(x))


model = DegradationSurrogate1DCNN().to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {n_params:,}')

# Shape verification
dummy = torch.zeros(4, 3, 60).to(device)
print(f'Input {dummy.shape}  →  Output {model(dummy).shape}')


Parameters: 196,225
Input torch.Size([4, 3, 60])  →  Output torch.Size([4, 1])


## 7 · Training with Early Stopping & LR Scheduling

- **ReduceLROnPlateau**: halves LR when val loss stalls for 8 epochs  
- **Early stopping**: stops training when val loss doesn't improve for 15 epochs  
- **Gradient clipping**: norm ≤ 1.0 prevents exploding gradients  
- **Best-weights restore**: saves best val-loss checkpoint in memory


In [62]:
EPOCHS   = 100
LR       = 1e-3
PATIENCE = 15

criterion = nn.HuberLoss(delta=0.5)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

train_losses, val_losses = [], []
best_val  = float('inf')
best_wts  = None
pat_cnt   = 0

for epoch in range(1, EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────────
    model.train()
    run = 0.0
    for Xb, yb in fit_loader:
        Xb, yb = Xb.to(device), yb.unsqueeze(-1).to(device)
        optimizer.zero_grad()
        loss = criterion(model(Xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        run += loss.item()
    t_loss = run / len(fit_loader)

    # ── Validate ───────────────────────────────────────────────────
    model.eval()
    vrun = 0.0
    with torch.no_grad():
        for Xb, yb in val_loader:
            Xb, yb = Xb.to(device), yb.unsqueeze(-1).to(device)
            vrun += criterion(model(Xb), yb).item()
    v_loss = vrun / len(val_loader)

    train_losses.append(t_loss)
    val_losses.append(v_loss)
    scheduler.step()

    # ── Early stopping ────────────────────────────────────────────
    if v_loss < best_val:
        best_val = v_loss
        best_wts = {k: v.clone() for k, v in model.state_dict().items()}
        pat_cnt  = 0
    else:
        pat_cnt += 1

    if epoch % 10 == 0 or epoch == 1:
        lr_now = optimizer.param_groups[0]['lr']
        print(f'Epoch {epoch:3d}/{EPOCHS}  '
              f'train={t_loss:.5f}  val={v_loss:.5f}  '
              f'lr={lr_now:.2e}  patience={pat_cnt}/{PATIENCE}')
    if pat_cnt >= PATIENCE:
        print(f'Early stop at epoch {epoch}')
        break

model.load_state_dict(best_wts)  # restore best
print(f'\nBest val MSE = {best_val:.6f}')


Epoch   1/100  train=0.08832  val=0.04617  lr=1.00e-03  patience=0/15
Epoch  10/100  train=0.01542  val=0.03503  lr=9.76e-04  patience=6/15
Early stop at epoch 19

Best val MSE = 0.013172


## 8 · Evaluation — MAE, RMSE, R² on All Splits

Primary-test and Secondary-test (Batch 3) numbers are the ones comparable to Severson 2019.


In [63]:
def evaluate(loader, name):
    model.eval()
    preds, actuals = [], []
    with torch.no_grad():
        for Xb, yb in loader:
            p = model(Xb.to(device)).cpu().squeeze(-1).numpy()
            preds.extend(p.tolist())
            actuals.extend(yb.numpy().tolist())
    p = denorm_y(np.array(preds))
    a = denorm_y(np.array(actuals))
    mae  = float(np.mean(np.abs(p - a)))
    rmse = float(np.sqrt(np.mean((p - a)**2)))
    r2   = float(r2_score(a, p))
    print(f'  {name:22s}  MAE={mae:.5f} Ah   RMSE={rmse:.5f} Ah   R²={r2:.4f}')
    return p, a

# Rebuild val loader from saved indices
val_X_arr = X_train_n[[i for i in val_ds.indices]]
val_y_arr = y_train_n[[i for i in val_ds.indices]]
val_eval_loader = DataLoader(
    TensorDataset(torch.FloatTensor(val_X_arr), torch.FloatTensor(val_y_arr)),
    batch_size=512, shuffle=False)

print('=' * 65)
print('EVALUATION RESULTS')
print('=' * 65)
p_val, a_val = evaluate(val_eval_loader, 'Validation (20% holdout)')
p_pri, a_pri = evaluate(pri_loader,      'Primary Test (Batch 1+2)')
p_sec, a_sec = evaluate(sec_loader,      'Secondary Test (Batch 3)')
print('=' * 65)


EVALUATION RESULTS
  Validation (20% holdout)  MAE=0.00676 Ah   RMSE=0.00952 Ah   R²=0.9712
  Primary Test (Batch 1+2)  MAE=0.01024 Ah   RMSE=0.01937 Ah   R²=0.8876
  Secondary Test (Batch 3)  MAE=0.01040 Ah   RMSE=0.01504 Ah   R²=0.8824


## 9 · Visualisations

In [64]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.suptitle('DegradationSurrogate1DCNN — Training & Evaluation', fontsize=13, fontweight='bold')

# Loss curves
ax = axes[0]
ax.plot(train_losses, color='steelblue', label='Train MSE')
ax.plot(val_losses,   color='orange',    label='Val MSE')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE (normalised)')
ax.set_title('Train / Val Loss')
ax.legend(); ax.grid(alpha=0.3)

for ax, (p, a, title, color) in zip(
        axes[1:],
        [(p_val, a_val, 'Validation', 'steelblue'),
         (p_pri, a_pri, 'Primary Test (Batch 1+2)', 'green'),
         (p_sec, a_sec, 'Secondary Test (Batch 3)', 'purple')]):
    lo = min(a.min(), p.min()) - 0.01
    hi = max(a.max(), p.max()) + 0.01
    ax.scatter(a, p, alpha=0.25, s=4, color=color)
    ax.plot([lo, hi], [lo, hi], 'r--', linewidth=1.5)
    r2 = r2_score(a, p)
    mae = np.mean(np.abs(p - a))
    ax.set_title(f'{title}\nR²={r2:.4f}  MAE={mae:.4f} Ah')
    ax.set_xlabel('Actual (Ah)'); ax.set_ylabel('Predicted (Ah)')
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.grid(alpha=0.3)

plt.tight_layout()
out_png = os.path.join(MODEL_DIR, 'surrogate_eval.png')
plt.savefig(out_png, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_png}')


Saved: /Users/mahizhan/Documents/Sem7/Github/data-driven-prediction-of-battery-cycle-life-before-capacity-degradation/Model/surrogate_eval.png


## 10 · Save Model + Normalisation Statistics

In [65]:
# Full checkpoint (model + normalisation stats for future inference)
full_path = os.path.join(MODEL_DIR, 'degradation_surrogate_1dcnn.pth')
torch.save({
    'model_state': model.state_dict(),
    'ch_mean':     ch_mean.tolist(),
    'ch_std':      ch_std.tolist(),
    'y_mean':      y_mean,
    'y_std':       y_std,
    'num_pts':     NUM_PTS,
    'input_shape': [3, NUM_PTS],
    'arch':        'DegradationSurrogate1DCNN',
}, full_path)
print(f'Full checkpoint: {full_path}')

# Backward-compatible weights-only save (keeps thermal agent working)
compat_path = os.path.join(MODEL_DIR, 'health_agent_1dcnn.pth')
torch.save(model.state_dict(), compat_path)
print(f'Compat weights:  {compat_path}')

print('\n--- Summary ---')
print(f'  Architecture : DegradationSurrogate1DCNN')
print(f'  Input        : (batch, 3, {NUM_PTS})  [V, I, T as 3 channels]')
print(f'  Parameters   : {sum(p.numel() for p in model.parameters()):,}')
print(f'  Best val MSE : {best_val:.6f}')


Full checkpoint: /Users/mahizhan/Documents/Sem7/Github/data-driven-prediction-of-battery-cycle-life-before-capacity-degradation/Model/degradation_surrogate_1dcnn.pth
Compat weights:  /Users/mahizhan/Documents/Sem7/Github/data-driven-prediction-of-battery-cycle-life-before-capacity-degradation/Model/health_agent_1dcnn.pth

--- Summary ---
  Architecture : DegradationSurrogate1DCNN
  Input        : (batch, 3, 60)  [V, I, T as 3 channels]
  Parameters   : 196,225
  Best val MSE : 0.013172


In [66]:
from copy import deepcopy

ENSEMBLE_SEEDS = [42, 7, 123, 256, 999]
ensemble_models = []

for s in ENSEMBLE_SEEDS:
    torch.manual_seed(s)
    np.random.seed(s)
    m = DegradationSurrogate1DCNN().to(device)
    opt = optim.Adam(m.parameters(), lr=LR, weight_decay=1e-5)
    sch = optim.lr_scheduler.ReduceLROnPlateau(opt, 'min', factor=0.5, patience=8)
    best_v, best_w, pat = float('inf'), None, 0
    for ep in range(1, EPOCHS + 1):
        m.train()
        for Xb, yb in fit_loader:
            Xb, yb = Xb.to(device), yb.unsqueeze(-1).to(device)
            opt.zero_grad(); loss = criterion(m(Xb), yb); loss.backward(); opt.step()
        m.eval(); vl = sum(criterion(m(Xb.to(device)), yb.unsqueeze(-1).to(device)).item()
                          for Xb, yb in val_loader) / len(val_loader)
        sch.step(vl)
        if vl < best_v: best_v = vl; best_w = deepcopy(m.state_dict()); pat = 0
        else: pat += 1
        if pat >= PATIENCE: break
    m.load_state_dict(best_w)
    ensemble_models.append(m)
    print(f'Seed {s}: best_val={best_v:.5f}')

def ensemble_predict(loader):
    all_preds = []
    for m in ensemble_models:
        m.eval()
        preds = []
        with torch.no_grad():
            for Xb, _ in loader:
                preds.extend(m(Xb.to(device)).cpu().squeeze(-1).tolist())
        all_preds.append(np.array(preds))
    return denorm_y(np.mean(all_preds, axis=0))  # averaged prediction

# Evaluate ensemble
for loader, name in [(pri_loader,'Primary Test'), (sec_loader,'Secondary Test')]:
    actuals = denorm_y(np.array([y for _, yb in loader for y in yb.tolist()]))
    preds   = ensemble_predict(loader)
    mae     = np.mean(np.abs(preds - actuals))
    r2      = r2_score(actuals, preds)
    print(f'{name:25s}  MAE={mae:.5f}  R²={r2:.4f}')


KeyboardInterrupt: 